<a id="introduction"></a>

# Problem Set 5 Solution: Autoencoders
## CHEME 5820 · Machine Learning and Artificial Intelligence for Engineers · Spring 2025

---

Handwritten digit images from the [MNIST database](https://en.wikipedia.org/wiki/MNIST_database) contain rich spatial structure — stroke patterns, curves, and loops that vary across writing styles yet remain recognisable as the same character. In this problem set solution, we implement and train a deterministic **Autoencoder (AE)** that learns to compress each 784-pixel image into a low-dimensional **bottleneck code** and then reconstruct it from that code alone.

The autoencoder is a direct neural-network analogue of the embedding models from the previous problem set. In CBOW and Skip-Gram, a weight matrix $\mathbf{W}_1$ maps a sparse one-hot word vector into a dense embedding, and $\mathbf{W}_2$ decodes that embedding back into a prediction. An autoencoder does the same thing for images — but with deeper, non-linear encoder and decoder networks, and with reconstruction error as the training signal instead of cross-entropy.

> __Learning Objectives__
>
> By the end of this problem set, you should be able to:
> * __Implement an encoder and decoder using Flux.jl:__ Write `encode` and `decode` functions that map inputs to a low-dimensional bottleneck and back, using `Chain` and `Dense` layers with ReLU and sigmoid activations.
> * __Implement and minimise a reconstruction loss:__ Compute mean-squared error between the input and its reconstruction, and write a Flux.jl training loop that minimises it using the Adam optimiser.
> * __Visualise and interpret the learned latent space:__ Encode images to their bottleneck codes, decode them back, and linearly interpolate between two codes to test whether the AE has learned a smooth latent geometry.

Let's get started!

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading the MNIST dataset, and setting up the required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of `Include.jl` in the notebook's global scope. `Include.jl` sets paths, loads required external packages, and includes `src/Types.jl` and `src/Compute.jl`, which define the `MyAEModel` type and the helper functions used throughout this notebook.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [Flux.jl](https://fluxml.ai/Flux.jl/stable/) for automatic differentiation and neural network layers, [MLDatasets.jl](https://juliaml.github.io/MLDatasets.jl/stable/) to load MNIST, and [Plots.jl](https://docs.juliaplots.org/stable/) for visualization. `Include.jl` loads these packages and includes `src/Types.jl` (defines `MyAEModel`) and `src/Compute.jl` (defines the helper functions below).

> __Helper functions (provided in `src/Compute.jl`)__
>
> * `build_ae_model(input_dim, hidden_dim, latent_dim)`: Constructs a `MyAEModel` with a symmetric encoder–decoder architecture. The encoder maps $D \to H \to H/2 \to L$ using ReLU activations with no output activation at the bottleneck; the decoder inverts this path and ends with a sigmoid to constrain reconstructions to $[0, 1]$.
> * `load_mnist_digit(digit; n_examples)`: Loads MNIST training images for one digit class and returns a $(784 \times N)$ Float32 matrix with pixel values in $[0, 1]$.
> * `show_image_grid(X; nrows, ncols)`: Displays columns of a $784 \times N$ matrix as a grid of 28×28 greyscale images.

The three implemented functions — `encode`, `decode`, and `reconstruction_loss` — are the core of the autoencoder forward pass. Let's define them in the Implementations section below.

### Implementations

The three functions below are the core of the autoencoder forward pass. All local function implementations are defined here so they are easy to find in one place.

> __`encode(model, x)`__
>
> Maps a data batch $\mathbf{x} \in \mathbb{R}^{D \times N}$ to bottleneck codes $\mathbf{z} \in \mathbb{R}^{L \times N}$ by passing `x` through `model.encoder`. The encoder is a three-layer `Chain` ($784 \to 256 \to 128 \to L$) with ReLU activations and no output activation at the bottleneck, leaving the codes unconstrained.

`decode` inverts this mapping, reconstructing pixel vectors from their codes:

> __`decode(model, z)`__
>
> Maps bottleneck codes $\mathbf{z} \in \mathbb{R}^{L \times N}$ back to pixel space $\hat{\mathbf{x}} \in \mathbb{R}^{D \times N}$ by passing `z` through `model.decoder`. The decoder is a three-layer `Chain` ($L \to 128 \to 256 \to 784$) ending with a sigmoid, which constrains reconstructions to $[0, 1]$ to match the normalised pixel range.

`reconstruction_loss` composes `encode` and `decode` to compute the training objective:

> __`reconstruction_loss(model, x)`__
>
> Computes the mean-squared reconstruction error: encode `x` to `z`, decode `z` to `x̂`, and return `mean(sum((x .- x̂).^2; dims=1))`. The inner `sum` accumulates squared pixel errors over the $D = 784$ dimensions for each example independently; the outer `mean` averages across the $N$ examples in the batch.

Let's implement the three functions.

In [ ]:
function encode(model::MyAEModel, x::AbstractMatrix)
    return model.encoder(x)       # D×N  →  L×N
end

function decode(model::MyAEModel, z::AbstractMatrix)
    return model.decoder(z)       # L×N  →  D×N  (sigmoid constrains output to [0, 1])
end

function reconstruction_loss(model::MyAEModel, x::AbstractMatrix)
    z  = encode(model, x)         # D×N → L×N
    x̂  = decode(model, z)         # L×N → D×N
    return mean(sum((x .- x̂).^2; dims=1))   # scalar MSE
end


### Constants
Let's store some constants that control the dataset, model architecture, and training schedule, etc. See the comment lines for explanations of each constant. You can change these values to experiment with different settings, but the provided values should work well for training a simple autoencoder on MNIST.

In [ ]:
DIGIT         = 3;       # MNIST digit class to model (0–9)
K             = 100;     # number of training examples
D             = 784;     # input dimension: 28 × 28 = 784 pixels
L             = 8;       # bottleneck (latent) dimension
H             = 256;     # hidden-layer width
LR            = 1f-3;    # Adam learning rate
NUM_EPOCHS    = 2_000;   # training epochs

Random.seed!(42);

___
### Background: Autoencoders

An autoencoder is a neural network trained to reproduce its own input at the output layer,
subject to passing through a low-dimensional **bottleneck**. It consists of two sub-networks:

| Component | Maps | Role |
|-----------|------|------|
| **Encoder** $f_\theta$ | $\mathbf{x}\in\mathbb{R}^D \to \mathbf{z}\in\mathbb{R}^L,\; L\ll D$ | compresses input to a compact code |
| **Decoder** $g_\phi$ | $\mathbf{z}\in\mathbb{R}^L \to \hat{\mathbf{x}}\in\mathbb{R}^D$ | reconstructs the input from the code |

Both are trained jointly to minimise the **reconstruction loss** over the training set:

$$\mathcal{L}(\theta,\phi) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - g_\phi(f_\theta(\mathbf{x}_i))\|_2^2$$

Because the only path from input to output passes through the $L$-dimensional bottleneck,
the encoder is forced to retain only the most important structure of the data — and the
decoder must learn to recover the full input from that compressed representation.

> __Connection to CBOW and Skip-Gram__
>
> You have already built models with this compress-then-reconstruct structure. In CBOW,
> the input weight matrix $\mathbf{W}_1$ acts as an encoder that maps a sparse one-hot
> word vector into a dense $d_h$-dimensional embedding, and $\mathbf{W}_2$ decodes that
> embedding back into a prediction over the vocabulary. An autoencoder is the same idea
> applied to continuous inputs (images) with deeper, non-linear encoder and decoder networks.

### Data Loading and Exploration

We train the autoencoder on $K = 100$ examples of MNIST digit **3** from the training split. Each 28×28 greyscale image is flattened to a 784-dimensional vector and stored as a column of the data matrix $\mathbf{X} \in \mathbb{R}^{784 \times K}$, with pixel values in $[0, 1]$.

> __`load_mnist_digit(DIGIT; n_examples = K)`__
>
> Queries `MLDatasets.MNIST(:train)` for all images of digit `DIGIT`, selects the first `K`, flattens each 28×28 array to a 784-element column vector, and returns a `(784 × K)` Float32 matrix. Pixel values are already normalised to $[0, 1]$ by MLDatasets.

Let's load the data, compute pixel statistics, and visualise a sample grid.

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
X = load_mnist_digit(DIGIT; n_examples=K);
println("Data matrix size: ", size(X));   # (784, 100)

> __What is going on in this code block?__
>
> `load_mnist_digit(DIGIT; n_examples=K)` (defined in `src/Compute.jl`) queries
> `MLDatasets.MNIST(:train)` for all images of the target digit, selects the first $K$,
> flattens each 28×28 array to a 784-element column vector, and returns the `(784 × K)`
> Float32 matrix `X`. Pixel values are already normalised to $[0,1]$ by MLDatasets.

With the data loaded, let's characterise the pixel value distribution of the training set.

> __Pixel statistics__
>
> The mean pixel value of MNIST digit images is well below 0.5 — most pixels are dark background — which explains why the initial reconstruction loss is large: the decoder must learn to reproduce many near-zero pixels.

Let's compute the mean and standard deviation of all pixel values in `X`.

In [ ]:
let
    μ_data = mean(X);
    σ_data = std(X);
    @printf("Mean pixel value: %.4f\n", μ_data);
    @printf("Std  pixel value: %.4f\n", σ_data);
end

> __Visualising the training data__
>
> Before building the model, let's look at a sample of the digit images we are working with. `show_image_grid` reshapes each 784-element column of `X` back to a 28×28 array and renders it as a greyscale heatmap, so we can verify the data loaded correctly and get an intuition for the variation in handwriting style across examples.

Let's display a 4×4 grid of training examples.

In [ ]:
show_image_grid(X; nrows=4, ncols=4)

___
## Task 1: Autoencoder Architecture

The autoencoder forward pass requires two operations. The **encoder** compresses each $D = 784$-pixel input to an $L = 8$-dimensional bottleneck code, and the **decoder** reconstructs the input from that code alone. Both are one-liners that call through the `Chain` networks stored in `model.encoder` and `model.decoder` — see the **Implementations** section in Setup above.

> __`build_ae_model(D, H, L)`__
>
> Constructs a `MyAEModel` with a symmetric architecture. The encoder compresses $784 \to 256 \to 128 \to L = 8$ using ReLU activations, with **no activation on the bottleneck** so that codes are unconstrained real numbers. The decoder inverts this path ($8 \to 128 \to 256 \to 784$) and ends with a **sigmoid** that constrains reconstructions to $[0, 1]$, matching the normalised pixel range.

Let's build the model and inspect its architecture.

In [ ]:
ae = build_ae_model(D, H, L);
println("AE created.")
println("  Encoder : ", ae.encoder)
println("  Decoder : ", ae.decoder)

> __What is going on in this code block?__
>
> `build_ae_model(D, H, L)` (in `src/Compute.jl`) constructs a `MyAEModel` with a
> symmetric architecture. The encoder compresses $784\to256\to128\to L=8$ using ReLU
> activations, with **no activation on the bottleneck** so that codes are unconstrained
> real numbers. The decoder inverts this path ($8\to128\to256\to784$) and ends with a
> **sigmoid** that constrains reconstructions to $[0,1]$, matching the normalised pixel range.

___
## Task 2: Reconstruction Loss and Training

`reconstruction_loss` is defined in the **Implementations** section of Setup above. It implements the mean-squared reconstruction loss:

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|_2^2
= \texttt{mean}(\texttt{sum}((\mathbf{X} - \hat{\mathbf{X}})^{\odot 2};\,\text{dims}=1))$$

Let's verify the loss on a small sample and then run the training loop.

In [ ]:
let
    x_test = X[:, 1:5]
    @printf("Initial reconstruction loss (untrained): %.4f\n", reconstruction_loss(ae, x_test))
end

> __What is going on in this code block?__
>
> `reconstruction_loss(ae, x_test)` runs the full encode–decode forward pass on `x_test` and returns the scalar MSE. `sum(...; dims=1)` sums the $D=784$ squared pixel errors for each example independently (producing a $1\times N$ row vector), and `mean(...)` averages across examples. Verifying the untrained loss is finite and positive confirms that `encode` and `decode` compose correctly through the model before we commit to a full training run.

With the forward pass verified, let's set up the optimiser and run the training loop.

> __Training with `Flux.withgradient` and `Flux.update!`__
>
> `Flux.withgradient(ae) do m ... end` returns both the scalar loss and a gradient tuple in a single forward–backward pass. Inside the `do` block, `m` is a differentiable handle to `ae` — calling `reconstruction_loss(m, X)` there lets Flux trace gradients through the full encode–decode chain and compute gradients with respect to all parameters in `model.encoder` and `model.decoder`. `Flux.update!` then applies the Adam step to every parameter at once.

Let's train the autoencoder for `NUM_EPOCHS = 2000` epochs and store the per-epoch loss in `losses`.

In [ ]:
# ── SOLUTION ─────────────────────────────────────────────────────────────────
opt_state = Flux.setup(Adam(LR), ae);
losses    = Float32[];

println("Training autoencoder...");
for epoch in 1:NUM_EPOCHS
    loss, grads = Flux.withgradient(ae) do m
        reconstruction_loss(m, X)
    end;
    Flux.update!(opt_state, ae, grads[1]);
    push!(losses, loss);
    epoch % 400 == 0 && @printf("  Epoch %4d | loss = %.4f\n", epoch, loss);
end
println("Training complete.");

Let's plot the per-epoch training loss to inspect how the autoencoder converged.

In [ ]:
plot(losses;
    xlabel="Epoch", ylabel="Reconstruction Loss (MSE)",
    title="Autoencoder Training", label="MSE",
    color=:steelblue, lw=2, framestyle=:box)

> __What do we observe?__
>
> The loss curve decreases steeply at first and then flattens as the encoder and decoder settle into a stable compressed representation. It does not reach zero — some reconstruction error is irreducible given the $L = 8$-dimensional bottleneck — but the final loss is well below the initial value, confirming that training succeeded.

___
## Task 3: Latent Space Analysis

After training, we evaluate what the autoencoder has learned. First, we compare original images to their reconstructions to gauge how much detail the $L = 8$-dimensional bottleneck retains. Then, we test whether the bottleneck is **smooth** by linearly interpolating between two training images in latent space — if the decoded path transitions gradually from one image to the other, the encoder has learned a geometrically meaningful representation.

> __Reconstruction: originals vs. decoded__
>
> We encode eight training images to their $L = 8$-dimensional bottleneck codes and decode back to pixel space. The left panel shows the originals; the right panel shows the reconstructions. Well-trained reconstructions should be recognisable as digit 3 and capture the overall stroke structure, though some blurring is expected given the 98× compression.

Let's compare the originals and their reconstructions.

In [ ]:
# Reconstruction: original vs AE output
let
    n_show  = 8;
    x_orig  = X[:, 1:n_show];
    z_orig  = encode(ae, x_orig);
    x_recon = decode(ae, z_orig);

    p_orig  = show_image_grid(x_orig;  nrows=2, ncols=4);
    p_recon = show_image_grid(x_recon; nrows=2, ncols=4);
    plot(p_orig, p_recon; layout=(1, 2), size=(700, 250),
         plot_title="Left: originals   Right: reconstructions")
end

> __What is going on in this code block?__
>
> We encode eight training images to their $L=8$-dimensional bottleneck codes, then decode
> back to pixel space. The reconstruction quality shows how much information the 8-dimensional
> bottleneck retains from the original 784-dimensional input. Well-trained reconstructions
> should be recognisable as digit 3 and capture the overall stroke structure, though some
> fine-grained detail is expected to be blurred by the compression.

Next, we test whether the encoder has learned a smooth latent space by interpolating between two training images in latent space.

> __Latent-space interpolation__
>
> A smooth latent space means that moving along a straight line between two codes $\mathbf{z}_1$ and $\mathbf{z}_2$ produces a coherent sequence of decoded images. We test this by computing $\mathbf{z}_\alpha = (1 - \alpha)\mathbf{z}_1 + \alpha\mathbf{z}_2$ for ten evenly-spaced values of $\alpha \in [0, 1]$ and decoding the full path in a single batched call.

Let's encode two images, interpolate between their codes, and display the decoded path.

In [ ]:
# Latent-space interpolation
# ── SOLUTION ─────────────────────────────────────────────────────────────────
let
    n_steps = 10;
    x1 = X[:, 1:1];   # first training image  (784 × 1)
    x2 = X[:, 6:6];   # sixth training image  (784 × 1)

    z1 = encode(ae, x1);   # L × 1
    z2 = encode(ae, x2);   # L × 1

    alphas = range(0f0, 1f0; length=n_steps);
    z_path = hcat([(1f0 - α) .* z1 .+ α .* z2 for α in alphas]...);  # L × n_steps
    x_path = decode(ae, z_path);                                       # D × n_steps

    show_image_grid(x_path; nrows=2, ncols=5)
end

> __What is going on in this code block?__
>
> We encode two training images to their bottleneck codes $\mathbf{z}_1$ and $\mathbf{z}_2$,
> then construct 10 intermediate codes by linear interpolation:
> $\mathbf{z}_\alpha = (1-\alpha)\mathbf{z}_1 + \alpha\mathbf{z}_2$ for
> $\alpha\in\{0,\,\tfrac{1}{9},\,\tfrac{2}{9},\,\ldots,\,1\}$.
> The 10 codes are decoded in a single batched call and displayed as a 2×5 grid. If the
> autoencoder has learned a smooth latent space, the frames should transition gradually
> from the first digit to the sixth; if the space is fragmented, intermediate frames may
> look like blurry or incoherent mixtures.

___
<a id="discussion"></a>

## Discussion
Use the results from Tasks 1–3 to answer the discussion questions below.

---

**DQ1: Reconstruction quality and bottleneck capacity.** The autoencoder compresses each $D = 784$-pixel image into an $L = 8$-dimensional bottleneck code — a 98× reduction. The encoder must decide what structure to keep and what to discard.

> __Strategy__: Examine the reconstruction panel from Task 3. Are the reconstructions sharp or blurry? Change `L` in the constants block to `2`, `4`, `16`, and `32`, re-run all cells, and compare reconstruction quality across these settings. What fine-grained information appears to be discarded first as the bottleneck shrinks?

Use the strategy above to formulate your answer below.

> __Answer__: Fill in your answer here.

---

**DQ2: Latent space smoothness.** Linear interpolation between two bottleneck codes $\mathbf{z}_1$ and $\mathbf{z}_2$ tests whether the autoencoder has learned a smooth, well-structured latent space — or whether nearby codes decode to very different images.

> __Strategy__: Examine the interpolation grid from Task 3. Does the sequence transition smoothly from one image to another, or do intermediate frames look incoherent? Try changing `L` and re-running the interpolation cell. What does a smooth (or fragmented) transition reveal about the geometry of the learned latent space?

Use the strategy above to formulate your answer below.

> __Answer__: Fill in your answer here.

---

**DQ3: Generative limits of the standard autoencoder.** A standard AE places no explicit constraint on the distribution of bottleneck codes — the encoder is free to use any region of $\mathbb{R}^L$ that minimises reconstruction loss.

> __Strategy__: Run `extrema(encode(ae, X))` to inspect the range of the learned codes. Then sample `z_rand = randn(Float32, L, 1)` and decode it with `show_image_grid(decode(ae, z_rand))`. Does the output look like a realistic digit? Explain why or why not, and describe what additional constraint would be needed to make the AE a proper generative model.

Use the strategy above to formulate your answer below.

> __Answer__: Fill in your answer here.

In [ ]:
# Set each flag to true — all discussion questions are answered in the solution
did_I_answer_DQ1 = true;
did_I_answer_DQ2 = true;
did_I_answer_DQ3 = true;

___
## Summary
In this problem set we implemented and trained a deterministic Autoencoder on MNIST digit-3 images, establishing the encoder–decoder pattern that underlies deeper generative models. The deterministic AE provides the foundation that the Variational Autoencoder extends by adding probabilistic structure to the bottleneck.

> __Key Takeaways__
>
> * **The autoencoder is CBOW for images:** Both models learn a low-dimensional embedding by training a compress-then-reconstruct pipeline — CBOW through a single linear layer, the AE through a deep non-linear encoder and decoder. In both cases, the bottleneck forces the model to capture the most salient structure of the input.
> * **MSE reconstruction loss drives encoder and decoder jointly:** Because `encode` and `decode` are composed inside a single `Flux.withgradient` call, the autoencoder gradient flows from the output error all the way back through both networks in one pass — exactly the backpropagation algorithm you saw in the feed-forward network lecture.
> * **The latent space has no guaranteed structure:** Without explicit regularisation, the bottleneck codes can take any values that minimise reconstruction loss, so sampling a random point in latent space may not decode to a realistic image. The Variational Autoencoder addresses this by adding a KL penalty that forces the latent codes to follow a standard normal distribution, making the space smooth and suitable for generation.

___